# 🧠 Build Your Own Brain — Session 2
### From search engine to answer machine

Last time you built the **search** half of RAG. Today we add the **answer** half, and then learn how to tell whether the whole thing is any good.

**You don't need to write code today.** Your job is to run cells, *predict* what will happen, and understand *why*. If you can explain how it works, you can build it again with any tool you like.

| Symbol | Meaning |
|---|---|
| ▶️ | Just run the cell |
| 🎛️ | A "dial" you're encouraged to change and re-run |
| 🌟 | Optional extra |

> **If something breaks:** read the red error (most are one-liners) → a `429` just means "slow down", and the code waits and retries by itself → still stuck? **Runtime → Restart session**, then **Run all** → raise your hand / traffic-light cup. 🟢 fine · 🟡 question · 🔴 stuck

## 🔧 Step 0 — Setup (▶️)

In [ ]:
!pip install -q -U google-genai pypdf requests
print("✅ Packages installed.")

### Your API key, and picking a model
Paste your key when asked (same key as last time). The next cell also **finds a working AI model for you automatically**. Google renames and retires models regularly, so instead of trusting one name, the notebook tries a short list and uses the first that works.

In [ ]:
from getpass import getpass
from google import genai
from google.genai import types

API_KEY = getpass("Paste your Gemini API key here and press Enter: ")
client = genai.Client(api_key=API_KEY)

EMBED_MODEL = "gemini-embedding-001"

# Sanity check: a bad key or no network should fail HERE, not three cells later.
_probe = client.models.embed_content(model=EMBED_MODEL, contents="hello")
print(f"✅ Embedding model works ({len(_probe.embeddings[0].values)} numbers per chunk).")

In [ ]:
# FACILITATORS: after your dry run, paste the model name that worked into the line below
# (for example GEN_MODEL_OVERRIDE = "gemini-2.5-flash-lite") so the whole room uses the same model.
GEN_MODEL_OVERRIDE = None

# Google retires and renames models over time, so instead of trusting one name we try a short
# list and use the first that works. "Flash-Lite" models have the most generous free limits.
GEN_CANDIDATES = [
    "gemini-flash-lite-latest",
    "gemini-3.1-flash-lite",
    "gemini-2.5-flash-lite",
    "gemini-flash-latest",
]


def pick_generation_model(client, override=None):
    names = [override] if override else GEN_CANDIDATES
    for name in names:
        try:
            client.models.generate_content(model=name, contents="Reply with the single word: OK")
            return name
        except Exception as e:
            msg = str(e)
            if "404" in msg or "NOT_FOUND" in msg or "not found" in msg.lower():
                print(f"  (model '{name}' isn't available, trying the next one...)")
                continue
            if "429" in msg or "RESOURCE_EXHAUSTED" in msg:
                return name  # the model exists, we're just being told to slow down
            raise  # something else (e.g. a bad API key): show the real error
    raise RuntimeError("None of the candidate models worked. Open https://ai.google.dev/gemini-api/docs/models, "
                       "copy a current Flash-Lite model name, and set GEN_MODEL_OVERRIDE above.")


GEN_MODEL = pick_generation_model(client, GEN_MODEL_OVERRIDE)
print(f"✅ Generation model: {GEN_MODEL}")

### Session 1 in one cell (▶️)
Everything you built last time — the loader, the chunker, the embedder, the retriever — lives in this one cell. **Just run it.** (Missed Session 1? This cell is all you need.)

In [ ]:
# ===== Session 1, in one cell. Just run it. =====
import re
import requests

# Loading and chunking text is FREE regardless of size -- no API call happens until
# the embedding step. These limits only control how many embedding *requests* we
# fire off live. 150,000 chars is roughly 330 chunks, i.e. ~4 API calls.
MIN_CHARS = 1500
MAX_CHARS = 150000
HEADERS = {"User-Agent": "GDG-HWU-RAG-Workshop/1.0 (student workshop, run in Colab)"}


def _extract_gdoc_id(url_or_id):
    match = re.search(r"/d/([a-zA-Z0-9_-]+)", url_or_id)
    return match.group(1) if match else url_or_id


def _load_wikipedia(title):
    resp = requests.get(
        "https://en.wikipedia.org/w/api.php",
        params={"action": "query", "prop": "extracts", "explaintext": 1,
                "redirects": 1, "titles": title, "format": "json"},
        headers=HEADERS, timeout=20,
    )
    resp.raise_for_status()
    page = next(iter(resp.json()["query"]["pages"].values()))
    if "missing" in page or not page.get("extract"):
        raise ValueError(f"No Wikipedia article found for '{title}'. Check the exact title.")
    text = page["extract"]
    if "may refer to" in text[:300]:
        raise ValueError(f"'{title}' is a disambiguation page -- use a more specific title.")
    return text


def _load_gdoc(url_or_id):
    doc_id = _extract_gdoc_id(url_or_id)
    resp = requests.get(f"https://docs.google.com/document/d/{doc_id}/export?format=txt",
                        headers=HEADERS, timeout=20)
    resp.raise_for_status()
    # A private doc doesn't error -- Google quietly returns a sign-in HTML page instead.
    if "<html" in resp.text[:500].lower():
        raise ValueError("Couldn't read that Google Doc. Set sharing to "
                         "'Anyone with the link can view' and try again.")
    return resp.text


def _load_pdf(path):
    from pypdf import PdfReader
    reader = PdfReader(path)
    text = "\n".join((page.extract_text() or "") for page in reader.pages)
    if not text.strip():
        raise ValueError("No text could be extracted. This is probably a scanned/image PDF, "
                         "which this workshop doesn't support -- use a text-based PDF.")
    return text


def _load_txt(path):
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        return f.read()


def _load_upload(_ignored=None):
    from google.colab import files  # only exists inside Colab
    print("Choose your .txt or .pdf file...")
    uploaded = files.upload()
    if not uploaded:
        raise ValueError("No file was uploaded.")
    name = next(iter(uploaded))
    return _load_pdf(name) if name.lower().endswith(".pdf") else _load_txt(name)


def load_text(source_type, source=None):
    """Universal loader: any source in, one clean string out, with a size check.

    source_type -> what `source` should be
      "wikipedia" -> exact article title
      "gdoc"      -> Google Doc link or ID (shared as 'Anyone with the link can view')
      "upload"    -> nothing (a file picker pops up; .txt or .pdf)
      "pdf"/"txt" -> a file path already in Colab's file panel
      "paste"     -> the raw text itself
      "demo"      -> nothing (the built-in demo text)
    """
    loaders = {
        "wikipedia": _load_wikipedia,
        "gdoc": _load_gdoc,
        "upload": _load_upload,
        "pdf": _load_pdf,
        "txt": _load_txt,
        "paste": lambda s: s,
        "demo": lambda _: DEMO_TEXT,
    }
    if source_type not in loaders:
        raise ValueError(f"Unknown source_type '{source_type}'. Choose one of: {list(loaders)}")

    text = loaders[source_type](source).strip()
    text = re.sub(r"\n{3,}", "\n\n", text)
    char_count = len(text)
    print(f"Loaded {char_count:,} characters (about {char_count // 450} chunks).")

    if char_count < MIN_CHARS:
        print(f"⚠️  That's short (under {MIN_CHARS:,} chars). Search works better with "
              f"more material -- consider a longer article or section.")
    if char_count > MAX_CHARS:
        text = text[:MAX_CHARS]
        print(f"⚠️  Trimmed from {char_count:,} to {MAX_CHARS:,} characters to stay inside "
              f"free-tier limits. Tip: pick ONE focused article, not a whole wiki or book.")
    return text

def chunk_text(text, chunk_size=500, overlap=50):
    if overlap >= chunk_size:
        raise ValueError("overlap must be smaller than chunk_size")
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size

        # Don't cut a word in half: pull `end` back to the last space.
        if end < len(text):
            last_space = text.rfind(" ", start, end)
            if last_space > start:
                end = last_space

        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)
        if end >= len(text):
            break

        # Step forward, but back up by `overlap` so neighbouring chunks share a little text...
        start = max(end - overlap, start + 1)
        # ...and nudge to the next space so the new chunk starts on a whole word.
        next_space = text.find(" ", start, end)
        if next_space != -1:
            start = next_space + 1
    return chunks

import time
import pickle
import os
import hashlib
import numpy as np

MAX_BATCH = 100  # Gemini's per-request limit for embed_content


def _fingerprint(texts, task_type):
    # md5 (not Python's hash()) so the fingerprint survives a Colab runtime restart
    h = hashlib.md5(task_type.encode())
    for t in texts:
        h.update(t.encode("utf-8"))
    return h.hexdigest()


def _call_embed(client, batch, task_type):
    """One API call, with automatic retries. Gives up loudly -- never silently."""
    config = types.EmbedContentConfig(task_type=task_type)
    last_error = None
    for attempt in range(5):
        try:
            result = client.models.embed_content(model=EMBED_MODEL, contents=batch, config=config)
            return [e.values for e in result.embeddings]
        except Exception as e:
            last_error = e
            wait = 5 * (2 ** attempt)  # 5, 10, 20, 40, 80 seconds (rate limits need real waits)
            print(f"  ⚠️ API error: {e}\n     Retrying in {wait}s (attempt {attempt + 1}/5)...")
            time.sleep(wait)
    raise RuntimeError(f"Embedding failed after 5 attempts: {last_error}")


def embed_texts(client, texts, checkpoint_path=None, delay_seconds=1,
                task_type="RETRIEVAL_DOCUMENT"):
    """Turn a list of texts into a matrix: one row (vector) per text, same order.
    Progress is saved after every batch, so re-running resumes instead of restarting."""
    fingerprint = _fingerprint(texts, task_type)
    all_embeddings = []

    if checkpoint_path and os.path.exists(checkpoint_path):
        try:
            with open(checkpoint_path, "rb") as f:
                saved = pickle.load(f)
            if saved.get("fingerprint") == fingerprint:
                all_embeddings = saved["embeddings"]
                print(f"Resuming from checkpoint: {len(all_embeddings)}/{len(texts)} already embedded.")
        except Exception:
            print("Ignoring unreadable checkpoint; starting fresh.")

    for i in range(len(all_embeddings), len(texts), MAX_BATCH):
        batch = texts[i:i + MAX_BATCH]
        vectors = _call_embed(client, batch, task_type)
        if len(vectors) != len(batch):
            # Misaligned embeddings would silently return the WRONG chunks -- so stop.
            raise RuntimeError(f"Sent {len(batch)} chunks but got {len(vectors)} embeddings back.")
        all_embeddings.extend(vectors)
        print(f"  Embedded {len(all_embeddings)}/{len(texts)} chunks...")

        if checkpoint_path:
            with open(checkpoint_path, "wb") as f:
                pickle.dump({"fingerprint": fingerprint, "embeddings": all_embeddings}, f)
        time.sleep(delay_seconds)

    matrix = np.array(all_embeddings)
    assert matrix.shape[0] == len(texts), "embedding count doesn't match chunk count"
    return matrix

def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-10)


def embed_query(client, query):
    # Questions use task_type RETRIEVAL_QUERY; chunks used RETRIEVAL_DOCUMENT.
    # The model is trained to match the two, which improves search quality.
    return np.array(_call_embed(client, [query], "RETRIEVAL_QUERY")[0])


def retrieve(client, query, chunks, chunk_embeddings, top_k=3):
    if len(chunks) != len(chunk_embeddings):
        raise ValueError(f"{len(chunks)} chunks but {len(chunk_embeddings)} embeddings -- "
                         f"re-run the chunking and embedding cells together.")
    query_embedding = embed_query(client, query)
    scores = [cosine_similarity(query_embedding, emb) for emb in chunk_embeddings]
    ranked = sorted(zip(chunks, scores), key=lambda x: x[1], reverse=True)
    return ranked[:top_k]


def show(results):
    for rank, (chunk, score) in enumerate(results, start=1):
        print(f"#{rank}  (similarity: {score:.3f})")
        print(chunk)
        print("---")

print("✅ Session 1 tools loaded: load_text, chunk_text, embed_texts, retrieve, show")

---
##  Part 1 — Today's test document
We'll use a **made-up** club handbook. That's on purpose: the AI has never seen it, so it's a *fair test* of whether the AI is using the documents or its own memory.

It's about 7,500 characters (~17 chunks). Run the next two cells: the first loads the handbook, the second builds the search engine from it, exactly like Session 1.

In [ ]:
# A made-up document, so the AI can't possibly have seen it before.
# (Every name, date, room and price below is invented for this workshop.)
HANDBOOK_TEXT = """DESERT FALCON CODE CLUB - MEMBER HANDBOOK

1. About the club
The Desert Falcon Code Club was founded in March 2019 by three second-year students who wanted a friendly place to build things together. The club's motto is "Ship it, then polish it." Our mascot is a robotic falcon called Zephyr, who appears on every poster, sticker and hoodie. The club is open to students from every programme, and no prior coding experience is needed to join. Our home base is the Maker Space (Room A0.14), where most build nights take place. The club is run entirely by student volunteers and is overseen by a faculty advisor, Dr. Sana Qureshi, who meets the committee once a month.

2. Membership tiers
There are three membership tiers. Fledgling membership is free and includes entry to the weekly Beginner Lab and access to the public Nest workspace channels. Falcon membership costs AED 40 per semester and adds equipment loans, voting rights in committee elections, and access to the private Nest project channels. Peregrine membership costs AED 120 per year and includes everything in Falcon plus priority registration for Falcon Hack 24, laptop loans, and a club hoodie in sand orange. Members can upgrade at any time by paying the difference; the amount already paid for Falcon is deducted from the Peregrine price. Joining mid-semester is allowed: the Falcon fee is reduced by AED 10 for each full month already missed, with a minimum payment of AED 10.

3. Leadership team
The 2026 committee has five members. The President is Layla Haddad, who chairs meetings and represents the club to the university. The Vice President for Projects is Omar Siddiqui, who assigns mentors to project teams. The Treasurer is Mei Tanaka, who approves all spending above AED 200. The Secretary is Rashid Al-Mansoori, who keeps minutes and manages room bookings. The Outreach Lead is Priya Nair, who runs social media and sponsor communication. Committee members serve one academic year. Elections are held in the second week of April, and any Falcon or Peregrine member may stand for a post after attending at least six sessions.

4. Weekly schedule
The Beginner Lab runs every Thursday at 4:15 pm in Room B3.07 and is designed for people who have never written code. Open Build Night runs every Monday at 6:30 pm in the Maker Space, where members work on their own projects with mentors on hand. The Interview Prep Circle meets every second Wednesday at 1:00 pm in Room C2.11 and covers CV reviews and mock technical interviews. Sessions are cancelled during exam weeks, and cancellations are announced in the Nest at least 24 hours in advance.

5. Events calendar
This year's flagship events are as follows. The Nest Warm-Up Jam on 18 October takes place in Room C1.02 and has a capacity of 40 people; it is a relaxed first-meeting build session. Falcon Hack 24 takes place on 22 and 23 November in the Atrium Hall with a capacity of 120 participants, teams of three or four, and a prize pool of AED 6,000. The Winter Data Night on 14 December is held in Room B2.05 with a capacity of 30 and focuses on data visualisation. The Spring Showcase on 9 April is held in Auditorium 1 with a capacity of 250 and gives every project team five minutes on stage. The Gulf Student Code Cup, an inter-university contest, takes place on 3 May at Harbour Institute; the club bus leaves from Gate 2 at 7:00 am and returns at about 8:00 pm.

6. Equipment loans
Falcon and Peregrine members may borrow one development board, either a Raspberry Pi 5 or an ESP32 kit, for up to 14 days. A refundable deposit of AED 100 is required for each board. Late returns are charged AED 5 per day, capped at AED 50 in total. If a board is lost or damaged beyond repair, the deposit is kept and the member pays any remaining replacement cost. Laptop loans are available only to Peregrine members: a laptop may be kept for 48 hours with a refundable deposit of AED 500. Fledgling members cannot borrow equipment but may use the boards in the Maker Space during Open Build Night.

7. Room booking
Room bookings are made through the booking form in the Nest and must be submitted at least 3 working days before the event. A booking can last a maximum of 2 hours unless the committee approves an extension. The Maker Space holds a maximum of 18 people at once, and food and drink are not allowed inside because of the electronics. Snacks are permitted in the Atrium Hall and in the Beginner Lab room, provided the organisers clear up afterwards. The Secretary confirms or rejects every booking within 2 working days.

8. Awards
Three awards are given each term. The Golden Commit Award goes to the member who gave the most helpful code review and comes with a AED 150 voucher. The Falcon Eye Award goes to the member who found the most interesting bug and comes with a AED 100 voucher. The Rookie of the Term award goes to the best new Fledgling member and comes with a free Falcon membership for the next semester. Winners are announced at the final Thursday session of each term.

9. Code of conduct
The club follows a three-strike system for breaches of the code of conduct. A first strike results in a verbal reminder from a committee member. A second strike results in a written warning and a 2-week event ban. A third strike results in suspension of membership for one semester. Harassment, discrimination and threats are treated more seriously and bypass the strike system: the member is suspended immediately while the committee reviews the case, and a decision is communicated within 5 working days. Members may appeal any decision to the faculty advisor within 7 days.

10. Sponsorship
The club welcomes sponsors at three levels. Bronze sponsorship is AED 2,000 per year and includes a logo on the club website. Silver sponsorship is AED 5,000 per year and adds a logo on the sleeve of the club hoodie. Gold sponsorship is AED 10,000 per year and adds a logo on the Falcon Hack 24 banner and a seat on the judging panel. Sponsorship income is divided as follows: 60% goes to events, 25% to equipment, and 15% to travel and competitions.

11. Travel policy
For approved external competitions the club covers 75% of the registration fee. Teams may have a maximum of four members and must submit a one-page plan to the Vice President for Projects at least 10 days before the registration deadline. Accommodation and meals are not covered. Every trip must have a trip leader who is a Peregrine member and who is responsible for attendance, transport and reporting back to the committee within a week.

12. Frequently asked questions
Can non-members attend Falcon Hack 24? Yes, but only as guests of a member, and no more than 20 guests are admitted in total. Do I lose my equipment deposit if I upgrade my membership? No, the deposit transfers automatically to your new tier. Are events open to students from other campuses? Only the Spring Showcase audience is open to other campuses; all other events are for registered members and their guests. Can the club store my project code? Projects remain the property of their authors, and the club only keeps a copy if the author agrees.

13. Privacy and photos
The club keeps member contact details for 2 years after a member leaves, after which they are deleted. At events, photographers use a wristband system: a green wristband means the person is happy to be photographed, and a red wristband means they must not appear in any photo or video. Photos that include green-wristband members may be used on club social media and the website. Members can ask for any photo of them to be removed within 3 working days by messaging the Outreach Lead.
"""

# "load_text('demo')" returns this document in Session 2.
DEMO_TEXT = HANDBOOK_TEXT

# A small exam for our search engine. "must_contain" = text that the RIGHT chunk has to contain.
DEMO_TESTS = [
    {"q": "How much is Falcon membership each semester?",                     "must_contain": ["AED 40 per semester"]},
    {"q": "Which room hosts the Nest Warm-Up Jam?",                           "must_contain": ["C1.02"]},
    {"q": "For how many days can I borrow a development board?",              "must_contain": ["14 days"]},
    {"q": "How much is the late fee per day for a development board?",        "must_contain": ["AED 5 per day"]},
    {"q": "Who is the club treasurer?",                                       "must_contain": ["Mei Tanaka"]},
    {"q": "What time does the bus to the Gulf Student Code Cup leave?",       "must_contain": ["7:00 am"]},
    {"q": "What happens when a member gets a second strike?",                 "must_contain": ["2-week event ban"]},
    {"q": "What colour is the Peregrine hoodie?",                             "must_contain": ["sand orange"]},
]

In [ ]:
handbook = load_text("demo")
chunks = chunk_text(handbook)
embeddings = embed_texts(client, chunks, checkpoint_path="handbook_checkpoint.pkl")
print(f"\n✅ Search engine ready: {len(chunks)} chunks, matrix shape {embeddings.shape}")

---
## 🔎 Part 2 — Retrieval is not an answer
Ask the search engine a question. Look at what comes back.

In [ ]:
# Retrieval alone gives back CHUNKS of text, not an answer.
question = "How much is the late fee per day for a development board?"
show(retrieve(client, question, chunks, embeddings, top_k=3))

The right information is in there, but it's **raw chunks of handbook**, not a clear answer. Nobody wants to read three paragraphs to learn "AED 5". We want a sentence.

---
## Part 3 — Augmentation + generation
A RAG prompt is just **text made of three parts**:

| Part | What it is |
|---|---|
| **Instructions** | Rules for the model ("use ONLY the context…", "say I don't know if it isn't there") |
| **Context** | The chunks we retrieved, numbered so the model can cite them |
| **Question** | What the user asked |

There's no magic: the model reads this text and continues it with an answer. This cell defines the pieces (▶️ run it, then read through it; **the `INSTRUCTIONS` text is 🎛️ yours to edit later**).

In [ ]:
import time

# ---- The instructions we give the model. This is just text: you can edit it! ----
INSTRUCTIONS = """You are a helpful assistant for a student society.
Answer the question using ONLY the context below.
If the context does not contain the answer, reply exactly: I don't know based on the documents.
Keep the answer to 1-3 sentences and mention which context number(s) you used, like [2]."""


def build_prompt(question, retrieved, instructions=None):
    """Glue together: instructions + the retrieved chunks + the question. That's all a RAG prompt is."""
    instructions = instructions or INSTRUCTIONS
    context = "\n\n".join(f"[{i}] {chunk}" for i, (chunk, score) in enumerate(retrieved, start=1))
    return f"{instructions}\n\nCONTEXT:\n{context}\n\nQUESTION: {question}\n\nANSWER:"


GEN_MIN_GAP = 6          # seconds between answers: the free tier allows only ~10 per minute
_last_generate = [0.0]


def _call_generate(client, prompt, temperature=0.2):
    """One call to the language model, with polite pacing and automatic retries."""
    config = types.GenerateContentConfig(temperature=temperature)  # low temperature = less creative, more faithful
    last_error = None
    for attempt in range(4):
        pause = GEN_MIN_GAP - (time.time() - _last_generate[0])
        if pause > 0:
            time.sleep(pause)
        try:
            _last_generate[0] = time.time()
            response = client.models.generate_content(model=GEN_MODEL, contents=prompt, config=config)
            return (response.text or "").strip() or "(The model returned an empty answer.)"
        except Exception as e:
            last_error = e
            wait = 10 * (2 ** attempt)   # 10, 20, 40, 80 seconds
            print(f"  ⚠️ API error: {e}\n     Retrying in {wait}s (attempt {attempt + 1}/4)...")
            time.sleep(wait)
    raise RuntimeError(f"Generation failed after 4 attempts: {last_error}")


def ask(client, question, chunks, chunk_embeddings, top_k=3, min_score=0.0,
        instructions=None, show_prompt=False):
    """The whole RAG loop: retrieve -> build prompt -> generate."""
    retrieved = retrieve(client, question, chunks, chunk_embeddings, top_k)

    # Guardrail: if even the BEST chunk scores below min_score, don't bother the model.
    if retrieved[0][1] < min_score:
        return {"answer": "I don't know based on the documents.", "sources": retrieved, "skipped_model": True}

    prompt = build_prompt(question, retrieved, instructions)
    if show_prompt:
        print("=========== THE PROMPT THE MODEL SEES ===========")
        print(prompt)
        print("=================================================\n")
    return {"answer": _call_generate(client, prompt), "sources": retrieved, "skipped_model": False}


def ask_without_rag(client, question):
    """The same model with NO documents: a closed-book exam."""
    return _call_generate(client, f"Answer in one short sentence: {question}")


def show_answer(result):
    print("ANSWER:", result["answer"])
    if result["skipped_model"]:
        print("(Top score was below min_score, so the model was never called.)")
    print("\nSOURCES USED:")
    for i, (chunk, score) in enumerate(result["sources"], start=1):
        print(f"  [{i}] (score {score:.3f}) {chunk[:90]}...")

### The showdown: closed-book vs open-book
Same AI model, same question. One version has no documents (a **closed-book exam**); the other gets the retrieved chunks (an **open-book exam**).

**Before you run it, predict:** will the plain model (A) invent a confident number, (B) say it doesn't know, or (C) get it right? Note your guess, then run.

The notebook also prints the exact prompt the model sees, so you can see Part 3 for real.

In [ ]:
# Closed-book (plain model) vs open-book (RAG). Same model, same question.
# TRY IT: before running, guess what the plain model will say!
question = "How much is the late fee per day if I return a borrowed development board late to the Desert Falcon Code Club?"

print("🙈 WITHOUT RAG (closed book):")
print(ask_without_rag(client, question))

print("\n📖 WITH RAG (open book):")
show_answer(ask(client, question, chunks, embeddings, top_k=3, show_prompt=True))

### 🔒 Grounding: why the instructions matter
Now a question that is **not** in the handbook. Same documents, same model, but two different sets of instructions. Watch what the "lax" version does.

In [ ]:
# The prompt matters. Same question that ISN'T in the handbook, two different instructions.
LAX_INSTRUCTIONS = "You are a helpful assistant. Use the context below to answer the question."
out_of_scope = "Does the Desert Falcon Code Club offer free pizza on Fridays?"

print("🔒 STRICT instructions (\"ONLY use the context... say I don't know\"):")
print(ask(client, out_of_scope, chunks, embeddings)["answer"])

print("\n🔓 LAX instructions (no 'only', no 'I don't know' rule):")
print(ask(client, out_of_scope, chunks, embeddings, instructions=LAX_INSTRUCTIONS)["answer"])

### A bouncer for bad questions
Instructions can fail, so we add a second guardrail that doesn't rely on the model at all: **if even the best chunk's similarity score is too low, don't call the model.** In the first cell, look at the gap between the two groups of scores. In the second, pick a cut-off between them. (Too high → it refuses good questions. Too low → junk gets through.)

In [ ]:
def top_scores(client, questions, chunk_embeddings):
    """Best similarity score for each question. One API call for ALL questions."""
    q_vecs = _call_embed(client, questions, "RETRIEVAL_QUERY")
    return [max(cosine_similarity(np.array(qv), emb) for emb in chunk_embeddings) for qv in q_vecs]


in_scope = ["How much is Falcon membership?", "Who is the treasurer?", "When is the Beginner Lab?"]
out_of_scope_qs = ["Who won the 2018 FIFA World Cup?", "How do I bake sourdough bread?", "What is the capital of Australia?"]

print("Best score for questions the handbook CAN answer:")
for q, s in zip(in_scope, top_scores(client, in_scope, embeddings)):
    print(f"  {s:.3f}  {q}")
print("\nBest score for questions the handbook can NOT answer:")
for q, s in zip(out_of_scope_qs, top_scores(client, out_of_scope_qs, embeddings)):
    print(f"  {s:.3f}  {q}")

In [ ]:
# Look at the gap between the two groups above, then pick a cut-off that sits between them.
MIN_SCORE = 0.0   # <- 🎛️ edit me! 0.0 means "never refuse". Try a number between the two groups.

show_answer(ask(client, "Who won the 2018 FIFA World Cup?", chunks, embeddings, min_score=MIN_SCORE))

---
## 📝 Part 4 — How do we know it's any good?
**Marking an open-book exam gives two separate marks**, because it can fail in two different places:

| Failure | What happened | Where to look |
|---|---|---|
| **Retrieval failure** | The right page was never handed over | The *sources* list |
| **Generation failure** | The right page was there, but the answer ignored or misread it | The *prompt* and *instructions* |
| **Knowledge gap** | The answer isn't in the documents at all | It *should* say "I don't know" |

To mark it fairly we write a **test set**: questions, plus a phrase the right chunk must contain. **hit@k** = "in what percentage of questions did the right chunk appear in the top *k* results?"

Run it. Which questions missed, and *why*? (Hint: look at what came back.)

In [ ]:
PLACEHOLDER = "WRITE"


def _first_hit_rank(ranked_indices, chunks, must_contain):
    """Walk down the ranked list; return the position (1 = top) of the first chunk
    that contains ALL the required words, or None if no retrieved chunk does."""
    for position, idx in enumerate(ranked_indices, start=1):
        text = chunks[idx].lower()
        if all(word.lower() in text for word in must_contain):
            return position
    return None


def retrieval_report(client, tests, chunks, chunk_embeddings, ks=(1, 3, 5), verbose=True):
    """Grade the SEARCH half: for each test question, did the right chunk come back in the top k?"""
    if any(t["q"].startswith(PLACEHOLDER) for t in tests):
        print("✏️  Replace the placeholder questions (the ones starting with WRITE) first!")
        return None

    q_vecs = _call_embed(client, [t["q"] for t in tests], "RETRIEVAL_QUERY")  # ONE call for all questions
    ranks = []
    for t, qv in zip(tests, q_vecs):
        scores = [cosine_similarity(np.array(qv), emb) for emb in chunk_embeddings]
        ranked_indices = np.argsort(scores)[::-1]
        ranks.append(_first_hit_rank(ranked_indices, chunks, t["must_contain"]))

    summary = {k: sum(1 for r in ranks if r is not None and r <= k) / len(tests) for k in ks}
    if verbose:
        print(f"{'Question':<62}{'Rank':>5}  Hit@{max(ks)}")
        for t, r in zip(tests, ranks):
            ok = "✅" if (r is not None and r <= max(ks)) else "❌"
            print(f"{t['q'][:60]:<62}{str(r) if r else '-':>5}  {ok}")
        print("\n" + "   ".join(f"hit@{k}: {summary[k]:.0%}" for k in ks))
    return summary


def answer_report(client, tests, chunks, chunk_embeddings, top_k=3):
    """Grade the WHOLE system: does the final answer contain the key fact? (Slow: one model call per question.)"""
    if any(t["q"].startswith(PLACEHOLDER) for t in tests):
        print("✏️  Replace the placeholder questions (the ones starting with WRITE) first!")
        return None
    correct = 0
    for t in tests:
        result = ask(client, t["q"], chunks, chunk_embeddings, top_k=top_k)
        ok = all(w.lower() in result["answer"].lower() for w in t["must_contain"])
        correct += ok
        print(f"{'✅' if ok else '❌'} {t['q']}\n     -> {result['answer'][:110]}")
    print(f"\nAnswers containing the key fact: {correct}/{len(tests)}")
    return correct / len(tests)


retrieval_report(client, DEMO_TESTS, chunks, embeddings)

### 🌟 Mark the final answers too (takes ~1 minute; the free tier is slow, so this is a good time to ask questions)

In [ ]:
# OPTIONAL (takes ~1 minute: the free tier only allows ~10 model calls per minute).
answer_report(client, DEMO_TESTS, chunks, embeddings)

---
## Part 5 — The dials
A RAG system is a control panel: **chunk size · overlap · how many chunks (k) · instructions · score cut-off · search method**. To improve it: change **one** dial, re-run the test set, see if the score moved. If you change two things at once, you can't know which one helped.

**Fair test:** you can only carry so much text into the exam hall, so every chunk size gets the same ~1,500-character *reading budget*: small chunks → fetch more of them, big chunks → fewer. Run it and look for a sweet spot.

In [ ]:
def chunk_size_experiment(client, text, tests, sizes=(200, 500, 1000, 2000), budget_chars=1500):
    """Re-chunk and re-embed the text at several chunk sizes and grade the search each time.

    FAIR TEST: every chunk size gets the same 'reading budget' (about budget_chars of text handed
    to the model). Small chunks => we fetch more of them; big chunks => we fetch fewer.
    (If we always fetched 3 chunks, huge chunks would win just because there are so few of them.)"""
    rows = []
    for size in sizes:
        k = max(1, round(budget_chars / size))
        size_chunks = chunk_text(text, chunk_size=size, overlap=size // 10)
        size_embeddings = embed_texts(client, size_chunks)
        summary = retrieval_report(client, tests, size_chunks, size_embeddings, ks=(k,), verbose=False)
        rows.append((size, len(size_chunks), k, summary[k]))
    print(f"\nSame reading budget (~{budget_chars} characters) for every row:")
    print(f"{'chunk size':>11} {'# chunks':>9} {'fetched':>8}   hit-rate")
    for size, n, k, score in rows:
        print(f"{size:>11} {n:>9} {k:>8}   {score:.0%}  {'█' * int(score * 20)}")
    return rows


# TRY IT: run this, then change `sizes` or `budget_chars` and run again.
# Which chunk size spends the reading budget best? Is there a sweet spot?
_ = chunk_size_experiment(client, handbook, DEMO_TESTS)

###  Level-up: hybrid search
Meaning-based search can miss exact things like names, codes and numbers. **Hybrid search** blends the meaning score with a simple "do the question's words literally appear?" score. This cell compares different blends on the same test set.

Other upgrades you could ask an AI assistant to help you build: rewrite the question before searching · re-rank the top results · cut chunks at sentence boundaries · use a vector database (e.g. Chroma) once you have thousands of chunks.

In [ ]:
# LEVEL-UP (optional): hybrid search = meaning score + simple keyword score.
import re

STOPWORDS = {"the", "a", "an", "is", "are", "of", "to", "for", "in", "on", "at", "do", "does", "how",
             "what", "who", "when", "where", "which", "much", "many", "can", "i", "my", "me", "and", "or", "it"}


def keyword_score(query, chunk):
    """Fraction of the question's important words that literally appear in the chunk (0 to 1)."""
    words = [w.strip(".") for w in re.findall(r"[a-z0-9.:]+", query.lower())]
    words = [w for w in words if w and w not in STOPWORDS]
    if not words:
        return 0.0
    chunk_lower = chunk.lower()
    return sum(w in chunk_lower for w in words) / len(words)


def score_chunks(query, query_vec, chunks, chunk_embeddings, keyword_weight=0.3):
    """Blend meaning (embedding similarity) with exact-word matching."""
    return [(1 - keyword_weight) * cosine_similarity(query_vec, emb) + keyword_weight * keyword_score(query, chunk)
            for chunk, emb in zip(chunks, chunk_embeddings)]


def hybrid_retrieve(client, query, chunks, chunk_embeddings, top_k=3, keyword_weight=0.3):
    scores = score_chunks(query, embed_query(client, query), chunks, chunk_embeddings, keyword_weight)
    ranked = sorted(zip(chunks, scores), key=lambda x: x[1], reverse=True)
    return ranked[:top_k]


def compare_keyword_weights(client, tests, chunks, chunk_embeddings, weights=(0.0, 0.2, 0.4, 0.6), k=3):
    """0.0 = pure meaning search (what we've used so far). Higher = more weight on exact words."""
    q_vecs = _call_embed(client, [t["q"] for t in tests], "RETRIEVAL_QUERY")
    print(f"{'keyword weight':>15}   hit@{k}")
    for w in weights:
        hits = 0
        for t, qv in zip(tests, q_vecs):
            scores = score_chunks(t["q"], np.array(qv), chunks, chunk_embeddings, w)
            rank = _first_hit_rank(np.argsort(scores)[::-1], chunks, t["must_contain"])
            hits += rank is not None and rank <= k
        print(f"{w:>15}   {hits / len(tests):.0%}")


compare_keyword_weights(client, DEMO_TESTS, chunks, embeddings)

# Other upgrades you could try next (not built here): ask the model to REWRITE the question before
# searching, RE-RANK the top 10 chunks with the model, cut chunks at sentence/paragraph boundaries,
# or move to a vector database such as Chroma once you have thousands of chunks.

---
## 🛠️ Part 6 — Your turn: your own topic
Same steps, your document. In order:
1. **Load it** — pick the source type (Wikipedia title, Google Doc link, your own upload, or pasted text). Keep it to one focused document.
2. **Write 5 test questions** — with a phrase the right chunk must contain. *Tip: pick answers that appear in exactly one place.*
3. **Grade your search engine** and write the score down.
4. **Ask it a real question** with the full RAG loop.
5. **Change one dial**, re-grade, and write down whether the score went up or down.

**Minimum to be "done":** index built · 5 tests written · a score recorded · one dial changed with the new score recorded.

In [ ]:
# ---- STEP 1: choose your source (same options as Session 1) ----
MY_SOURCE_TYPE = "wikipedia"   # "wikipedia", "gdoc", "upload", "paste", "txt", "pdf"
MY_SOURCE = "Your topic here"  # article title / gdoc link / (ignored for "upload") / text / path
# ----------------------------------------------------------------

my_text = load_text(MY_SOURCE_TYPE, MY_SOURCE)
my_chunks = chunk_text(my_text)
my_embeddings = embed_texts(client, my_chunks, checkpoint_path="my_checkpoint.pkl")
print(f"\n✅ Your search engine is ready: {len(my_chunks)} chunks.")

**Writing good test questions:** a ❌ on *every* question usually means your `must_contain` phrases don't match the text exactly (check spelling, capitals and punctuation). A ✅ on *every* question usually means the questions are too easy. Try rewording one so it doesn't copy words from the document.

In [ ]:
# ---- STEP 2: write a mini exam for YOUR topic (5 questions) ----
# "q" = the question.  "must_contain" = a word or phrase that the RIGHT chunk must contain.
# Tip: pick answers that appear in exactly one place in your document.
MY_TESTS = [
    {"q": "WRITE YOUR QUESTION 1 HERE", "must_contain": ["a word from the answer"]},
    {"q": "WRITE YOUR QUESTION 2 HERE", "must_contain": ["a word from the answer"]},
    {"q": "WRITE YOUR QUESTION 3 HERE", "must_contain": ["a word from the answer"]},
    {"q": "WRITE YOUR QUESTION 4 HERE", "must_contain": ["a word from the answer"]},
    {"q": "WRITE YOUR QUESTION 5 HERE", "must_contain": ["a word from the answer"]},
]

# ---- STEP 3: grade your search engine ----
retrieval_report(client, MY_TESTS, my_chunks, my_embeddings)

In [ ]:
# ---- STEP 4: ask your own system a question, using the full RAG loop ----
MY_QUESTION = "WRITE A QUESTION ABOUT YOUR TOPIC HERE"

if MY_QUESTION.startswith("WRITE"):
    print("✏️  Write your own question first!")
else:
    show_answer(ask(client, MY_QUESTION, my_chunks, my_embeddings, top_k=3))

# ---- STEP 5: improve it! Change ONE dial at a time, then re-run Step 3 to see if your score moved:
#   * chunk size:  my_chunks = chunk_text(my_text, chunk_size=300, overlap=30)   (then re-embed, and re-run Step 3!)
#   * top_k:       top_k=5
#   * instructions (edit INSTRUCTIONS in the prompt cell above)

---
## 🎉 You've now met every part of a RAG system
**load → chunk → embed → retrieve → build the prompt → generate → grade → adjust the dials.**

**Before you leave**
- Write down your *before* and *after* scores (we'll collect them).
- Save this notebook: *File → Save a copy in Drive*.

**For Session 3 (optional): Demo Day.** Finish your app, polish your 5 test questions (they become your efficiency score), and prepare a 60-second pitch: *what is it, who is it for, and what's the best (or funniest) question it answered?*